## Spring Thaw Accessor

### Fetching and Visualizing Steamboat Springs, CO boundary
Grabbing boundary from OSM, checking and obtaining type, bbox and doing a quick visualization.

In [ ]:
import osmnx
import geopandas as gpd

# Grabbing boundary from OSM
gdf = osmnx.geocoder.geocode_to_gdf("Steamboat Springs, Colorado") # Fetches Steamboat Springs, CO boundary into a gdf

# Checking geometry type of gdf, and obtaining that geometry
print(gdf.geom_type)
geom = gdf.geometry[0]
print(geom)
print(f'GDF CRS is: ', {gdf.crs})

# Reproject prior to buffer
gdf = gdf.to_crs(epsg=26913)

# Buffer the boundary by .35 km (350m)...started with 250m until all appropriate areas had SNOWDAS values. Edge cells with real values were being clipped to '0' SWE
gdf.geometry = gdf.buffer(350)

# Obtain bbox from gdf
bounds = gdf.bounds
minx = bounds.iloc[0]['minx']
miny = bounds.iloc[0]['miny']
maxx = bounds.iloc[0]['maxx']
maxy = bounds.iloc[0]['maxy']

print(f"Bounding box for Steamboat Springs, CO boundary: {minx}, {miny}, {maxx}, {maxy}")

# Reproject back to 4326
gdf = gdf.to_crs(epsg=4326)

# Exporting to a GeoJSON
gdf.to_file(r'data/processed/steamboat_springs_boundary.geojson', driver="GeoJSON")

# gdf.geometry[0] #quick viz
m = gdf.explore()
m

0    Polygon
dtype: object
POLYGON ((-106.886846 40.507395, -106.886418 40.506932, -106.886018 40.506654, -106.885778 40.506473, -106.885349 40.506142, -106.885046 40.506031, -106.884879 40.50597, -106.883096 40.506038, -106.882489 40.506211, -106.882049 40.506337, -106.881803 40.506013, -106.881505 40.505329, -106.881439 40.505382, -106.88105 40.505704, -106.881037 40.505715, -106.880977 40.505678, -106.880711 40.505514, -106.880399 40.505323, -106.879631 40.50505, -106.879248 40.504913, -106.879041 40.50484, -106.878912 40.504807, -106.878639 40.504737, -106.87845 40.504688, -106.878465 40.504637, -106.878476 40.504601, -106.878474 40.50452, -106.878498 40.504488, -106.878531 40.504447, -106.878508 40.504334, -106.878497 40.504279, -106.878486 40.50422, -106.878406 40.503821, -106.878397 40.503777, -106.878385 40.503716, -106.878364 40.50361, -106.878352 40.503559, -106.878333 40.503474, -106.878266 40.503453, -106.878217 40.503438, -106.878134 40.50343, -106.878051 40.503423, -106.8

### Downloading SNODAS Raster
Download, reproject (if needed), clip raster and create vector grid. 
SNODAS raster is in .tar format, and inside is a .dat.gz folder.

In [19]:
import shutil, tarfile, gzip, urllib.request, os

# Variable Setup
target_dir = os.path.join("data", "raw")
if not os.path.exists(target_dir):
    os.makedirs(target_dir)

url = 'https://noaadata.apps.nsidc.org/NOAA/G02158/masked/2024/04_Apr/SNODAS_20240412.tar'
tar_filepath = os.path.join(target_dir, 'SNODAS_20240412.tar')
tar_pre = 'us_ssmv11034'
tar_suf = '.dat.gz'

print("Starting download...")

# Download tar file from URL
urllib.request.urlretrieve(url, tar_filepath)

# Open .tar file to peek inside
with tarfile.open(tar_filepath, 'r') as tar:
    # Get list of all files inside
    all_files = tar.getnames()

    found_file = None

    # Look for the specific .dat.gz file
    for name in all_files:
        if name.startswith(tar_pre) and name.endswith(tar_suf):
            found_file = name
            print('Target file found. Extracting...')
            # Extract the file
            tar.extract(found_file, path=target_dir)
            break

# Unzip .gz part to get the .dat
if found_file:
    gz_full_path = os.path.join(target_dir, found_file)
    output_dat_file = gz_full_path.replace(".gz", "")
    
    print(f"Unzipping to {output_dat_file}...")

    with gzip.open(gz_full_path, 'rb') as f_in:
        with open(output_dat_file, 'wb') as f_out:
            shutil.copyfileobj(f_in, f_out)

    print(".dat file is ready")
else:
    print("Could not find file")


Starting download...
Target file found. Extracting...
Unzipping to data\raw\us_ssmv11034tS__T0001TTNATS2024041205HP001.dat...
.dat file is ready


C:\Users\DJ\AppData\Local\Temp\ipykernel_20672\2127196138.py:31: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tar.extract(found_file, path=target_dir)


### Converting SNODAS to GeoTiff


In [20]:
from osgeo import gdal, osr
import numpy as np
import rasterio

sno_file = 'snowdas_20240412.tif'

def convert_snodas_to_tif(input_dat, output_tif):
    
    output_processed = r'data\processed'
    os.makedirs(output_processed, exist_ok=True)

    snow_target_dir = os.path.join(output_processed, output_tif)
    
    # Grid size
    cols = 6935
    rows = 3351

    # Bounds for computing the Geotransform
    wmost = -124.73333333333333
    emost = -66.94166666666667
    nmost = 52.87500000000000
    smost = 24.95000000000000

    # Math for Geotransform
    lon_span = emost - wmost
    lat_span = nmost - smost
    deg_per_col = lon_span / cols
    deg_per_row = lat_span / rows

    # Read the binary data
    raw_data = np.fromfile(input_dat, dtype='>i2').reshape((rows,cols))

    # Create the GeoTIFF driver
    driver = gdal.GetDriverByName('GTiff')
    dataset = driver.Create(snow_target_dir, cols, rows, 1, gdal.GDT_Int16)

    # Define Geotransform
    dataset.SetGeoTransform([wmost, deg_per_col, 0, 
                              nmost, 0, -deg_per_row])

    # Define Projection
    srs = osr.SpatialReference()
    srs.ImportFromEPSG(4326)
    dataset.SetProjection(srs.ExportToWkt())

    # Write data to band 1
    band = dataset.GetRasterBand(1)
    band.WriteArray(raw_data)
    band.SetNoDataValue(-9999)

    # Flush to disk
    dataset = None
    print(f"Successfully created {snow_target_dir}")

# Usage
convert_snodas_to_tif(output_dat_file, sno_file)

# SNODAS Metadata checks
sno_file_path = os.path.join("data", "processed", sno_file)

sno_dataset = rasterio.open(sno_file_path)
print(f"\nSNODAS data is {sno_dataset.width} rows by {sno_dataset.height} columns.")
print(f"The current CRS is {sno_dataset.crs}")
print(f"The bounds are as follows: {sno_dataset.bounds}")
print(f"The raster is type: {sno_dataset.dtypes}")
print(f"The raster has following summary stats: {sno_dataset.statistics(1)}")
sno_dataset.close()
print("Raster has been closed...")

Successfully created data\processed\snowdas_20240412.tif

SNODAS data is 6935 rows by 3351 columns.
The current CRS is EPSG:4326
The bounds are as follows: BoundingBox(left=-124.73333333333333, bottom=24.95, right=-66.94166666666668, top=52.875)
The raster is type: ('int16',)
The raster has following summary stats: Statistics(min=-32768.0, max=32521.0, mean=-1652.6159916966399, std=4335.227692317536)
Raster has been closed...


### Clipping and de-scaling SNODAS Raster

- Opening the SNODAS raster with rasterio
- Clipping it using the Steamboat Springs boundary
- Applying the scale factor (divide by 1000)
- Writing the clipped, scaled result to a new GeoTIFF

In [21]:
from rasterio.mask import mask
import fiona

clip_shape = gdf.geometry.values # Steamboat springs boundary
scale_factor = 1000.0 # See SNODAS Userguide for information

# Apply the mask
with rasterio.open(sno_file_path) as src:
    out_image, out_transform = mask(src, clip_shape, crop=True)
    out_meta = src.meta.copy()

# Convert to float32 first, then modify the values
out_image_scaled = out_image.astype('float32') / scale_factor

# Replace negative values with 0
out_image_scaled[out_image_scaled < 0] = 0

# Update metadata for floats. Change dtype to float32
out_meta.update({
    "driver": "GTiff",
    "height": out_image_scaled.shape[1],
    "width": out_image_scaled.shape[2],
    "transform": out_transform,
    "dtype": 'float32',
    "nodata": np.nan
})

# Save result
processed_path = os.path.join("data", "processed", "clipped_snodas.tif")

with rasterio.open(processed_path, "w", **out_meta) as dest:
    dest.write(out_image_scaled)
src.close()

### Grid Creation and Raster Sampling to Grid  
geotransform is [minx, pixel_width, 0, maxy, 0, -pixel_height]

For each cell at position (row, col), you can calculate its corner coordinates:

- Top-left x = minx + (col * pixel_width)
- Top-left y = maxy + (row * (-pixel_height))
- Bottom-right x = top-left x + pixel_width
- Bottom-right y = top-left y + (-pixel_height)


You need two nested loops: one for rows, one for columns. For each cell:

- Calculate the four corner coordinates
- Create a Shapely polygon from those corners
- Get the SWE value from the raster at that location
- Store the polygon and value together

In [22]:
from shapely.geometry import Polygon

# Open raster
with rasterio.open(processed_path) as src:
    data = src.read(1)
    geotransform = src.transform

    # Extract tranform values
    minx = geotransform.c
    pixel_width = geotransform.a
    maxy = geotransform.f
    pixel_height = geotransform.e

    # Get raster dimensions
    rows, cols = data.shape

    # Blank lists to store geometries and values
    geometries = []
    swe_values = []

    # Creating grid using nested for loops
    for row in range(rows):
        for col in range(cols):
            # Calc corner coords
            min_x = minx + (col * pixel_width)
            max_x = min_x + pixel_width
            max_y = maxy + (row * pixel_height)
            min_y = max_y + pixel_height

            # Create polygon
            corners = [
                (min_x, max_y),
                (max_x, max_y),
                (max_x, min_y),
                (min_x, min_y)
            ]
            polygon = Polygon(corners)

            # Obtain SWE value
            swe_value = data[row, col]

            geometries.append(polygon)
            swe_values.append(swe_value)

    # Create GeoDataFrame
    gdf_grid = gpd.GeoDataFrame({
        'swe': swe_values,
        'geometry': geometries
    }, crs=src.crs)

# Save to GeoJSON
gdf_grid.to_file("data/processed/snowdas_grid.geojson", driver="GeoJSON")
print(f"Grid created with {len(gdf_grid)} cells")


Grid created with 168 cells


In [23]:
# Exploring the FEMA shapefiles
# from os import listdir
# from os.path import isfile, join
# files = [f for f in listdir(r'data\raw\08107C_20251126') if isfile(join(r'data\raw\08107C_20251126', f))] # Listing all the files in the shapefile directory
# print(files)

# Exploring the shapefile 'S_FLD_HAZ_AR.shp' as this is the one we are interested in
flood_zones = gpd.read_file(f'data/raw/08107C_20251126/S_FLD_HAZ_AR.shp') 

print(flood_zones.columns,'\n')
print(flood_zones.geom_type.unique(),'\n')
print(flood_zones.crs,'\n')
print(flood_zones.bounds,'\n')
print(flood_zones.head(),'\n')
print(flood_zones['FLD_ZONE'].unique(),'\n', flood_zones['ZONE_SUBTY'].unique())



Index(['DFIRM_ID', 'VERSION_ID', 'FLD_AR_ID', 'STUDY_TYP', 'FLD_ZONE',
       'ZONE_SUBTY', 'SFHA_TF', 'STATIC_BFE', 'V_DATUM', 'DEPTH', 'LEN_UNIT',
       'VELOCITY', 'VEL_UNIT', 'AR_REVERT', 'AR_SUBTRV', 'BFE_REVERT',
       'DEP_REVERT', 'DUAL_ZONE', 'SOURCE_CIT', 'geometry'],
      dtype='object') 

['Polygon'] 

EPSG:4269 

           minx       miny        maxx       maxy
0   -106.881026  40.268469 -106.831824  40.288209
1   -106.810959  40.445347 -106.792085  40.448079
2   -106.931208  40.250071 -106.883393  40.269266
3   -107.277882  40.494065 -107.267902  40.495340
4   -107.271827  40.495366 -107.266851  40.497608
..          ...        ...         ...        ...
620 -107.265553  40.492321 -107.263092  40.493005
621 -107.268391  40.494319 -107.267794  40.494950
622 -107.266707  40.494559 -107.265878  40.495268
623 -107.267287  40.490628 -107.264659  40.491551
624 -107.442626  39.918698 -106.625364  41.003424

[625 rows x 4 columns] 

  DFIRM_ID VERSION_ID  FLD_AR_ID STUDY_TYP 

In [28]:
# Filtering the shapefile to be only zones A, AE, AO. Clipping, reprojecting and exporting to GeoJSON.

a_zones = flood_zones[flood_zones['FLD_ZONE'].isin(['A', 'AE', 'AO'])]
a_zones = a_zones.to_crs(epsg=4326) # Reprojecting to 4326 to match rest of project

# Clipping to Steamboat Springs boundary
a_zones_clipped = gpd.clip(a_zones, gdf)

# Exporting to geojson
a_zones_clipped.to_file('data/processed/a_fld_zones_area.geojson',driver='GeoJSON')

In [39]:
# Extracting ALL OSM building footprints from Steamboat Springs, CO using OSMNX
import osmnx as ox

tags = {"building": True}

bldg_gdf = ox.features_from_place("Steamboat Springs, Colorado, USA", tags)

# Verify buildings projection, geom type, size
print("CRS of the data is ", bldg_gdf.crs)
print("Data has ", len(bldg_gdf), " features")
print("Geometry types of dataset is " , bldg_gdf.geom_type.unique())

bldg_gdf.to_file('data/processed/stmbt_sprgs_bldgs.geojson',driver='GeoJSON')


CRS of the data is  epsg:4326
Data has  5795  features
Geometry types of dataset is  ['Point' 'Polygon']


### DuckDB 



In [69]:
import duckdb

# Load spatial extension and the 'ST_Read_Multi from community exentions 
con = duckdb.connect()
con.execute("INSTALL spatial; LOAD spatial;")

# Ingest GeoJSON files into dbs
tbl_flood = con.sql(f"SELECT * FROM ST_Read('data/processed/a_fld_zones_area.geojson')")
tbl_snowdas = con.sql(f"SELECT * FROM ST_Read('data/processed/snowdas_grid.geojson')")
tbl_bldgs = con.sql(f"SELECT * FROM ST_Read('data/processed/stmbt_sprgs_bldgs.geojson')")

# Sanity check - view counts in tables
con.sql("SELECT count (*) FROM tbl_flood").show()
con.sql("SELECT count (*) FROM tbl_snowdas").show()
con.sql("SELECT count (*) FROM tbl_bldgs").show()

# Column information for tables
print("columns from tbl_flood: ", tbl_flood.columns)
print("columns from tbl_snowdas: ", tbl_snowdas.columns)
print("columns from tbl_bldgs: ", tbl_bldgs.columns)

┌──────────────┐
│ count_star() │
│    int64     │
├──────────────┤
│          162 │
└──────────────┘

┌──────────────┐
│ count_star() │
│    int64     │
├──────────────┤
│          168 │
└──────────────┘

┌──────────────┐
│ count_star() │
│    int64     │
├──────────────┤
│         5795 │
└──────────────┘

columns from tbl_flood:  ['DFIRM_ID', 'VERSION_ID', 'FLD_AR_ID', 'STUDY_TYP', 'FLD_ZONE', 'ZONE_SUBTY', 'SFHA_TF', 'STATIC_BFE', 'V_DATUM', 'DEPTH', 'LEN_UNIT', 'VELOCITY', 'VEL_UNIT', 'AR_REVERT', 'AR_SUBTRV', 'BFE_REVERT', 'DEP_REVERT', 'DUAL_ZONE', 'SOURCE_CIT', 'geom']
columns from tbl_snowdas:  ['swe', 'geom']
columns from tbl_bldgs:  ['element', 'id', 'building', 'ele', 'gnis:feature_id', 'is_in:city', 'is_in:state', 'is_in:state_code', 'name', 'amenity', 'parking', 'source', 'brand', 'brand:wikidata', 'tourism', 'access', 'fee', 'addr:city', 'addr:housenumber', 'addr:postcode', 'addr:state', 'addr:street', 'website', 'addr:country', 'historic', 'office', 'addr:unit', 'opening

In [85]:
# DuckDB - Spatial join > Querying for buildings that are within flood zone polygons

bldgs_in_flood = con.sql("SELECT DISTINCT buildings.id, buildings.geom FROM tbl_bldgs AS buildings JOIN tbl_flood AS flood ON ST_Intersects(buildings.geom, flood.geom)")
bldgs_in_flood.show()

┌───────────┬───────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────

In [108]:
# DuckDB - Spatial join > Joining 525 buildings from previous query to the grid cells, grouping by building ID and average SWE values
bldgs_grid_swe = con.sql("SELECT buildings.id, AVG(snowdas.swe) AS SWE_Average, buildings.geom FROM bldgs_in_flood AS buildings JOIN tbl_snowdas AS snowdas ON ST_Intersects(buildings.geom, snowdas.geom) GROUP BY buildings.id, buildings.geom")
bldgs_grid_swe.show()

┌────────────┬────────────────────┬─────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────

In [ ]:
# DuckDB - Creating vulnerability scores and categories. Export as GeoJSON
bldgs_grid_swe_vulnerability = con.sql("SELECT id, SWE_Average, ((SWE_Average - 0) / (32.512 - 0) * 100) AS normalized_score, CASE WHEN normalized_score < 25 THEN 'Low Risk' WHEN normalized_score < 50 THEN 'Medium Risk' WHEN normalized_score < 75 THEN 'High Risk' ELSE 'Very High Risk' END AS risk_category, geom FROM bldgs_grid_swe")
bldgs_grid_swe_vulnerability.show()



┌────────────┬─────────────┬────────────────────┬────────────────┬────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────┐
│     id     │ SWE_Average │  normalized_score  │ risk_category  │                                                                                                                                                                                              

AttributeError: This relation does not contain a column by the name of 'to_file'

In [124]:
# Exporting to a GeoJSON... DUCKDB > PD > GPD > GEOJSON
from shapely import wkt

export = con.sql("SELECT id, SWE_Average, normalized_score, risk_category, ST_AsText(geom) AS wkt_geometry FROM bldgs_grid_swe_vulnerability")
df_for_gpd_export = export.df()
df_for_gpd_export['geometry'] = df_for_gpd_export['wkt_geometry'].apply(wkt.loads) # Convert WKT strings to shapely geometry objects
gdf_for_export = gpd.GeoDataFrame(df_for_gpd_export, geometry='geometry')
gdf_for_export = gdf_for_export.drop('wkt_geometry', axis=1)


gdf_for_export.to_file('data/processed/bldgs_grid_swe_vulnerability.geojson', driver='GeoJSON')